In [8]:
import pulp
import numpy as np
from typing import Any

In [9]:
# Define constants
#

DELTA_T = 0.25  # 15 min in Hours
N = 96  # Number of horizon time steps

BAT_CAP = 5.12*4                # kWh
BAT_P_MAX = 4.0*3               # kW
ETA_CH = np.sqrt(0.95) * 0.96   # 95% BESS round-trip efficiency * 96% inverter power factor
ETA_DIS = np.sqrt(0.95) * 0.96  # 95% BESS round-trip efficiency * 96% inverter power factor
SOC_MIN = 0.1 * BAT_CAP
SOC_MAX = 0.9 * BAT_CAP


In [ ]:
def optimize_day(prices_epex, pv, load, soc0):

    # Preise in ct/kWh laut vkw.at am 22.12.2025
    price_sell = prices_epex - 0.6
    price_buy  = prices_epex + 1.44

    # Gewinn Maximierungsproblem
    model = pulp.LpProblem("DayAheadOpt", pulp.LpMaximize)
    T = range(N)

    # Variablen
    p_ch   = pulp.LpVariable.dicts("p_ch", T, 0, BAT_P_MAX)
    p_dis  = pulp.LpVariable.dicts("p_dis", T, 0, BAT_P_MAX)
    soc    = pulp.LpVariable.dicts("soc", T, SOC_MIN, SOC_MAX)

    p_sell = pulp.LpVariable.dicts("p_sell", T, 0)
    p_buy  = pulp.LpVariable.dicts("p_buy", T, 0)

    # Binärvariable gegen gleichzeitiges Laden/Entladen
    y = pulp.LpVariable.dicts("y", T, 0, 1, cat="Binary")

    # SOC Dynamik
    for t in T:
        if t == 0:
            model += soc[t] == soc0 + DELTA_T * (
                ETA_CH * p_ch[t] - p_dis[t] / ETA_DIS
            )
        else:
            model += soc[t] == soc[t-1] + DELTA_T * (
                ETA_CH * p_ch[t] - p_dis[t] / ETA_DIS
            )

    # Lade-/Entlade-Exklusivität
    for t in T:
        model += p_ch[t]  <= BAT_P_MAX * y[t]
        model += p_dis[t] <= BAT_P_MAX * (1 - y[t])

    # Leistungsbilanz
    for t in T:
        model += (
            pv[t] + p_buy[t] + p_dis[t]
            ==
            load[t] + p_sell[t] + p_ch[t]
        )

    # Zielfunktion: Erlös – Kosten
    model += pulp.lpSum(
        (price_sell[t] * p_sell[t]
         - price_buy[t] * p_buy[t]) * DELTA_T
        for t in T
    )

    model.solve(pulp.PULP_CBC_CMD(msg=False))

    return {
        "soc":    np.array([soc[t].value() for t in T]),
        "p_ch":   np.array([p_ch[t].value() for t in T]),
        "p_dis":  np.array([p_dis[t].value() for t in T]),
        "p_sell": np.array([p_sell[t].value() for t in T]),
        "p_buy":  np.array([p_buy[t].value() for t in T]),
    }


In [11]:
prices_epex = np.random.uniform(5, 30, N)
pv = np.random.uniform(0, 3, N)
load = np.random.uniform(1, 5, N)
soc0 = 0.5 * BAT_CAP

optimize_day(prices_epex, pv, load, soc0)


{'soc': array([ 8.4603647,  5.2541824,  2.048    ,  2.048    ,  2.048    ,
         4.8550768,  7.6621535,  7.6621535,  7.1262271,  6.9427915,
         3.7366091,  6.5436859,  3.3375036,  6.1445803,  8.9516571,
        11.758734 , 14.565811 , 17.372887 , 14.166705 , 16.973782 ,
        13.767599 , 10.561417 , 13.368494 , 16.175571 , 12.969388 ,
        12.365626 , 15.172702 , 11.96652  , 10.455893 ,  7.2497102,
        10.056787 , 12.863864 , 15.670941 , 12.464758 ,  9.2585759,
         6.0523935,  8.8594703,  5.6532879,  8.4603647,  5.2541824,
         2.048    ,  4.8550768,  7.6621535, 10.46923  ,  7.263048 ,
         4.0568656,  6.8639424,  6.8639424,  9.6710192,  9.6710192,
        12.478096 ,  9.2719136, 12.065653 ,  8.8594703,  5.6532879,
         8.4603647,  5.2541824,  2.048    ,  4.8550768,  2.8462112,
         2.8462112,  5.6532879,  8.4603647,  5.2541824,  2.048    ,
         4.8550768,  7.6621535, 10.46923  ,  9.7856038, 12.592681 ,
        12.592681 , 12.592681 , 11.673575

In [5]:
SOC = 10.0  # Start-SOC

for k in range(N_sim_steps):

    prices_h = prices[k : k+H]
    pv_h     = pv_forecast(k, H)
    load_h   = load_forecast(k, H)

    model = build_model(prices_h, pv_h, load_h, SOC)
    solver.solve(model)

    # Nur Schritt 0 umsetzen
    p_ch  = pyo.value(model.p_ch[0])
    p_dis = pyo.value(model.p_dis[0])

    # System simulieren
    SOC = SOC + DELTA_T * (ETA_CH * p_ch - p_dis / ETA_DIS)

    log_step(k, SOC, p_ch, p_dis)


NameError: name 'N_sim_steps' is not defined